# Displaying Random Image Captions for Manual Inspection

This script is designed to facilitate the manual evaluation of image captions by displaying images alongside their corresponding AI-generated descriptions. For each sampled haunted place, the script determines which ZIP archive the corresponding image is stored in based on its ID range (e.g., hpimgs_0-1999.zip, hpimgs_2000-3999.zip, etc.). Special handling is included for images within the 8000–8999 range, which are stored under a subfolder with a different naming convention. If the image is not already extracted, the script unzips only the necessary file to avoid decompressing entire archives. Finally, the image is displayed alongside its associated caption in a Jupyter Notebook using IPython.display, allowing for efficient visual inspection and qualitative evaluation of caption accuracy.

In [ ]:
import zipfile
import os
import pandas as pd
from IPython.display import Image, display

# Paths
ZIP_DIR = "/Users/serafinasmith/dsci_550_a1/data/generated_images"
EXTRACT_DIR = "/Users/serafinasmith/dsci_550_a1/data/generated_images"
TSV_PATH = "/Users/serafinasmith/dsci_550_a1/data/processed/haunted_places_features_added_v2.tab"

# Load the dataset
df = pd.read_csv(TSV_PATH, sep="\t")

# Sample 100 random rows
sample_df = df[df["Haunted_Places_Id"].between(0, 8999)].sample(n=100, random_state=42)

for idx, row in sample_df.iterrows():
    haunted_id = int(row["Haunted_Places_Id"])
    filename = f"hpimg_{haunted_id}.png"
    extract_path = os.path.join(EXTRACT_DIR, filename)

    # Determine which zip the image is in
    if 0 <= haunted_id < 2000:
        zip_name = "hpimgs_0-1999.zip"
    elif 2000 <= haunted_id < 4000:
        zip_name = "hpimgs_2000-3999.zip"
    elif 4000 <= haunted_id < 8000:
        zip_name = "hpimgs_4000-7999.zip"
    elif 8000 <= haunted_id < 9000:
        zip_name = "hpimgs_8000-8999.zip"
    else:
        continue  # Skip out-of-range

    zip_path = os.path.join(ZIP_DIR, zip_name)

    # Only extract if not already present
    if not os.path.exists(extract_path):
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            if zip_name == "hpimgs_8000-8999.zip":
                original_name = f"HauntedImages_8000/haunted_{haunted_id}.png"
                if original_name in zip_ref.namelist():
                    zip_ref.extract(original_name, path=EXTRACT_DIR)
                    os.rename(
                        os.path.join(EXTRACT_DIR, original_name),
                        extract_path
                    )
                else:
                    print(f"Image not found in zip: {original_name}")
                    continue
            else:
                if filename in zip_ref.namelist():
                    zip_ref.extract(filename, path=EXTRACT_DIR)
                else:
                    print(f"Image not found in zip: {filename}")
                    continue

    # Display the image and caption
    # print(f"Haunted Place ID: {haunted_id}")
    # display(Image(filename=extract_path, width=300))
    # print(f"Caption: {row['Image_Caption']}")
    # print("-" * 80)

# Analyzing Word Frequency Trends in AI-Generated Captions
This script performs a word frequency analysis on the AI-generated image captions to uncover common patterns and linguistic trends. First, it filters out any rows in the dataset with missing captions to ensure accurate analysis. It then uses scikit-learn’s CountVectorizer to tokenize the remaining captions while excluding common English stopwords. Each unique word is counted across the corpus of captions, and the total word frequency is calculated. The script compiles these frequencies into a DataFrame and computes the percentage each word contributes to the overall word count. After sorting the words by frequency, it displays the top 20 most frequently occurring words along with their relative percentages. To visually illustrate these findings, it generates a bar chart showing the frequency of these top words.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import matplotlib.pyplot as plt
import pandas as pd

# Drop missing captions
df_captions = df.dropna(subset=['Image_Caption'])

# Vectorize captions to count word frequencies
vectorizer = CountVectorizer(stop_words='english')
X = vectorizer.fit_transform(df_captions['Image_Caption'])

# Get word frequencies
word_freq = X.toarray().sum(axis=0)
words = vectorizer.get_feature_names_out()

# Create DataFrame of word frequencies
word_freq_df = pd.DataFrame(list(zip(words, word_freq)), columns=['word', 'freq'])

# Calculate total word count
total_words = word_freq_df['freq'].sum()

# Add percentage column
word_freq_df['percent'] = 100 * word_freq_df['freq'] / total_words

# Sort and get top 20
top_words_df = word_freq_df.sort_values(by='freq', ascending=False).head(20)

# # Display table of frequencies and percentages
# print("\nTop 20 Most Frequent Words:\n")
# print(top_words_df.to_string(index=False, formatters={
#     'percent': '{:.2f}%'.format
# }))

# # Plot the top words
# top_words_df.plot(kind='bar', x='word', y='freq', legend=False)
# plt.title('Top 20 Most Common Words in Captions')
# plt.ylabel('Frequency')
# plt.xlabel('Word')
# plt.xticks(rotation=45)
# plt.tight_layout()
# plt.show()